# Phase 4 - Layer D: Operational Scheduling Constraints

This notebook enriches the raw geometric visibility windows from Layer C with simple operational realism.

The satellite orbits and gateway coordinates are grounded in public data from the earlier layers. The operational fields in this notebook are synthetic because public Starlink gateway maintenance schedules, antenna reservations, and reconfiguration logs are not generally available. The synthetic rules are intentionally simple, deterministic, and documented so they can be reviewed or replaced later.

The first operational model assumes each gateway behaves as a single dish-like resource: one active contact at a time, a fixed reconfiguration time between contacts, and reproducible blackout windows that represent maintenance or local unavailability.

## Configuration

All Layer D files are kept in this `add-scheduling-data` folder. The notebook reads only the reviewed Layer C visibility-window CSV and writes the Layer D operational assumptions and enriched windows back to this folder.

In [13]:
from pathlib import Path
import hashlib
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = ["pandas"]
missing_packages = [
    package_name
    for package_name in REQUIRED_PACKAGES
    if importlib.util.find_spec(package_name) is None
]

if missing_packages:
    print(f"Installing missing packages into this notebook kernel: {missing_packages}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print("Dependency installation finished. If imports still fail, restart the kernel and run the notebook again.")

import pandas as pd

DEFAULT_ANTENNA_TYPE = "dish"
DEFAULT_STATION_CAPACITY = 1
RECONFIGURATION_TIME_SECONDS = 180
BLACKOUTS_PER_STATION = 2
BLACKOUT_DURATION_MINUTES = 30
BLACKOUT_RANDOM_SEED = 20260531
BLACKOUT_REASON = "synthetic_maintenance"

DATA_DIR = Path("../../data/dataset/starlink")
DATA_DIR.mkdir(parents=True, exist_ok=True)
VISIBILITY_INPUT_CSV = DATA_DIR / "starlink_visibility_windows.csv"
BLACKOUT_OUTPUT_CSV = DATA_DIR / "synthetic_gateway_blackouts.csv"
ENRICHED_OUTPUT_CSV = DATA_DIR / "starlink_visibility_windows_operational.csv"

print(f"Notebook Python executable: {sys.executable}")
print(f"Layer D data directory: {DATA_DIR.resolve()}")
print(f"Visibility input: {VISIBILITY_INPUT_CSV.resolve()}")
print(f"Blackout output: {BLACKOUT_OUTPUT_CSV.resolve()}")
print(f"Enriched output: {ENRICHED_OUTPUT_CSV.resolve()}")

Notebook Python executable: c:\Users\felip\anaconda3\python.exe
Layer D data directory: C:\Users\felip\Documents\Polimi\Semester 3\SE2 Quantum Satisfiability Project\satelite-optimization-quantum-annealing\PoliMi-SWE2-research-project\data\dataset\starlink
Visibility input: C:\Users\felip\Documents\Polimi\Semester 3\SE2 Quantum Satisfiability Project\satelite-optimization-quantum-annealing\PoliMi-SWE2-research-project\data\dataset\starlink\starlink_visibility_windows.csv
Blackout output: C:\Users\felip\Documents\Polimi\Semester 3\SE2 Quantum Satisfiability Project\satelite-optimization-quantum-annealing\PoliMi-SWE2-research-project\data\dataset\starlink\synthetic_gateway_blackouts.csv
Enriched output: C:\Users\felip\Documents\Polimi\Semester 3\SE2 Quantum Satisfiability Project\satelite-optimization-quantum-annealing\PoliMi-SWE2-research-project\data\dataset\starlink\starlink_visibility_windows_operational.csv


## Load Raw Visibility Windows

Layer D starts from the raw geometric visibility windows generated in Layer C. Times are parsed as UTC timestamps so blackout overlaps and availability checks are reliable.

In [14]:
def load_visibility_windows(visibility_csv_path: Path) -> pd.DataFrame:
    """Load and validate the Layer C visibility-window CSV.

    Parameters
    ----------
    visibility_csv_path:
        Path to the raw visibility-window CSV generated by Layer C.

    Returns
    -------
    pandas.DataFrame
        Visibility windows with UTC start and end timestamps.
    """
    if not visibility_csv_path.exists():
        raise FileNotFoundError(f"Missing Layer C visibility CSV: {visibility_csv_path}")

    windows_df = pd.read_csv(visibility_csv_path)
    required_columns = [
        "window_id",
        "satellite_name",
        "norad_id",
        "station_id",
        "station_name",
        "start_time",
        "end_time",
        "duration_seconds",
        "max_elevation_deg",
        "source_epoch",
    ]
    missing_columns = [column for column in required_columns if column not in windows_df.columns]
    if missing_columns:
        raise ValueError(f"Missing required Layer C columns: {missing_columns}")

    windows_df = windows_df.copy()
    windows_df["start_time"] = pd.to_datetime(windows_df["start_time"], utc=True)
    windows_df["end_time"] = pd.to_datetime(windows_df["end_time"], utc=True)

    invalid_time_order = (windows_df["end_time"] <= windows_df["start_time"]).sum()
    if invalid_time_order:
        raise ValueError(f"Layer C input has {invalid_time_order} windows with invalid time ordering.")

    return windows_df


visibility_windows = load_visibility_windows(VISIBILITY_INPUT_CSV)

print(f"Visibility windows loaded: {len(visibility_windows)}")
print(f"Unique stations: {visibility_windows['station_id'].nunique()}")
print(f"Unique satellites: {visibility_windows['norad_id'].nunique()}")
visibility_windows.head(10)

Visibility windows loaded: 4091
Unique stations: 50
Unique satellites: 100


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch
0,VIS_000001,STARLINK-1296,45412,GATEWAY_0001,"Adelanto, CA",2026-05-31 00:00:00+00:00,2026-05-31 00:06:00+00:00,360,56.652,2026-06-17T20:32:56Z
1,VIS_000002,STARLINK-1068,44772,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,17.071,2026-06-18T04:25:21Z
2,VIS_000003,STARLINK-1277,45369,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,10.446,2026-06-17T23:46:21Z
3,VIS_000004,STARLINK-1300,45374,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:03:00+00:00,180,38.754,2026-06-18T01:45:28Z
4,VIS_000005,STARLINK-1167,45071,GATEWAY_0003,"Akita, Japan",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,11.375,2026-06-18T02:34:02Z
5,VIS_000006,STARLINK-1154,45198,GATEWAY_0003,"Akita, Japan",2026-05-31 00:00:00+00:00,2026-05-31 00:04:00+00:00,240,21.188,2026-06-18T04:49:37Z
6,VIS_000007,STARLINK-1269,45197,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:05:00+00:00,300,33.113,2026-06-17T14:51:53Z
7,VIS_000008,STARLINK-1302,45375,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:02:00+00:00,120,20.103,2026-06-17T23:12:53Z
8,VIS_000009,STARLINK-1297,45395,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,10.072,2026-06-18T05:00:33Z
9,VIS_000010,STARLINK-1350,45535,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:03:00+00:00,180,13.082,2026-06-17T12:25:52Z


## Define Station Availability and Blackouts

Every station is available for the full Layer C horizon except during synthetic blackout windows. The blackout generator uses a hash of the station ID, blackout index, and seed, so the same inputs always produce the same blackout table.

In [15]:
def deterministic_fraction(*parts: object) -> float:
    """Create a stable pseudo-random fraction in the range [0, 1).

    Parameters
    ----------
    *parts:
        Values that uniquely define the deterministic draw.

    Returns
    -------
    float
        Stable fraction derived from a SHA-256 digest.
    """
    text = "|".join(str(part) for part in parts)
    digest = hashlib.sha256(text.encode("utf-8")).hexdigest()
    return int(digest[:12], 16) / float(16**12)


def build_station_table(windows_df: pd.DataFrame) -> pd.DataFrame:
    """Build one operational resource row for each gateway station.

    Parameters
    ----------
    windows_df:
        Visibility windows containing station IDs and station names.

    Returns
    -------
    pandas.DataFrame
        Station-level operational assumptions for Layer D.
    """
    stations_df = (
        windows_df[["station_id", "station_name"]]
        .drop_duplicates()
        .sort_values(["station_id", "station_name"])
        .reset_index(drop=True)
    )
    stations_df["antenna_type"] = DEFAULT_ANTENNA_TYPE
    stations_df["station_capacity"] = DEFAULT_STATION_CAPACITY
    stations_df["reconfiguration_time_seconds"] = RECONFIGURATION_TIME_SECONDS
    return stations_df


def generate_blackout_windows(
    stations_df: pd.DataFrame,
    horizon_start: pd.Timestamp,
    horizon_end: pd.Timestamp,
    blackouts_per_station: int,
    blackout_duration_minutes: int,
    seed: int,
) -> pd.DataFrame:
    """Generate deterministic synthetic blackout windows for each station.

    Parameters
    ----------
    stations_df:
        Station-level operational table.
    horizon_start:
        Start of the operational horizon.
    horizon_end:
        End of the operational horizon.
    blackouts_per_station:
        Number of blackout windows generated for each station.
    blackout_duration_minutes:
        Fixed duration of each blackout window.
    seed:
        Stable seed included in each hash-based draw.

    Returns
    -------
    pandas.DataFrame
        Blackout table with station IDs, times, durations, and reasons.
    """
    if blackouts_per_station < 0:
        raise ValueError("blackouts_per_station cannot be negative.")
    if blackout_duration_minutes <= 0:
        raise ValueError("blackout_duration_minutes must be positive.")

    horizon_seconds = int((horizon_end - horizon_start).total_seconds())
    blackout_seconds = blackout_duration_minutes * 60
    if blackouts_per_station and blackout_seconds * blackouts_per_station > horizon_seconds:
        raise ValueError("Requested blackout windows do not fit inside the operational horizon.")

    segment_seconds = horizon_seconds // max(1, blackouts_per_station)

    records = []
    for station in stations_df.itertuples(index=False):
        for blackout_index in range(blackouts_per_station):
            segment_start_offset = blackout_index * segment_seconds
            segment_end_offset = min((blackout_index + 1) * segment_seconds, horizon_seconds)
            latest_offset_in_segment = max(segment_start_offset, segment_end_offset - blackout_seconds)
            available_offset_span = max(0, latest_offset_in_segment - segment_start_offset)

            fraction = deterministic_fraction(seed, station.station_id, blackout_index)
            offset_seconds = segment_start_offset + int(fraction * available_offset_span)
            blackout_start = horizon_start + pd.Timedelta(seconds=offset_seconds)
            blackout_end = blackout_start + pd.Timedelta(seconds=blackout_seconds)

            records.append(
                {
                    "blackout_id": f"BLK_{station.station_id}_{blackout_index + 1:02d}",
                    "station_id": station.station_id,
                    "station_name": station.station_name,
                    "blackout_start": blackout_start.isoformat(),
                    "blackout_end": blackout_end.isoformat(),
                    "duration_seconds": blackout_seconds,
                    "reason": BLACKOUT_REASON,
                    "source": "deterministic synthetic Layer D assumption",
                }
            )

    return pd.DataFrame.from_records(records)


horizon_start = visibility_windows["start_time"].min().floor("min")
horizon_end = visibility_windows["end_time"].max().ceil("min")
stations = build_station_table(visibility_windows)
blackout_windows = generate_blackout_windows(
    stations,
    horizon_start,
    horizon_end,
    BLACKOUTS_PER_STATION,
    BLACKOUT_DURATION_MINUTES,
    BLACKOUT_RANDOM_SEED,
)

print(f"Operational horizon start: {horizon_start.isoformat()}")
print(f"Operational horizon end: {horizon_end.isoformat()}")
print(f"Stations with operational assumptions: {len(stations)}")
print(f"Synthetic blackout windows: {len(blackout_windows)}")
blackout_windows.head(10)

Operational horizon start: 2026-05-31T00:00:00+00:00
Operational horizon end: 2026-05-31T04:00:00+00:00
Stations with operational assumptions: 50
Synthetic blackout windows: 100


,blackout_id,station_id,station_name,blackout_start,blackout_end,duration_seconds,reason,source
0,BLK_GATEWAY_0001_01,GATEWAY_0001,"Adelanto, CA",2026-05-31T00:59:04+00:00,2026-05-31T01:29:04+00:00,1800,synthetic_maintenance,deterministic synthetic Layer D assumption
1,BLK_GATEWAY_0001_02,GATEWAY_0001,"Adelanto, CA",2026-05-31T02:55:36+00:00,2026-05-31T03:25:36+00:00,1800,synthetic_maintenance,deterministic synthetic Layer D assumption
2,BLK_GATEWAY_0002_01,GATEWAY_0002,"Aerzen, Germany",2026-05-31T01:09:06+00:00,2026-05-31T01:39:06+00:00,1800,synthetic_maintenance,deterministic synthetic Layer D assumption
3,BLK_GATEWAY_0002_02,GATEWAY_0002,"Aerzen, Germany",2026-05-31T02:35:32+00:00,2026-05-31T03:05:32+00:00,1800,synthetic_maintenance,deterministic synthetic Layer D assumption
4,BLK_GATEWAY_0003_01,GATEWAY_0003,"Akita, Japan",2026-05-31T01:23:08+00:00,2026-05-31T01:53:08+00:00,1800,synthetic_maintenance,deterministic synthetic Layer D assumption
5,BLK_GATEWAY_0003_02,GATEWAY_0003,"Akita, Japan",2026-05-31T02:50:17+00:00,2026-05-31T03:20:17+00:00,1800,synthetic_maintenance,deterministic synthetic Layer D assumption
6,BLK_GATEWAY_0004_01,GATEWAY_0004,"Akron, OH",2026-05-31T01:13:21+00:00,2026-05-31T01:43:21+00:00,1800,synthetic_maintenance,deterministic synthetic Layer D assumption
7,BLK_GATEWAY_0004_02,GATEWAY_0004,"Akron, OH",2026-05-31T02:24:24+00:00,2026-05-31T02:54:24+00:00,1800,synthetic_maintenance,deterministic synthetic Layer D assumption
8,BLK_GATEWAY_0005_01,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-05-31T01:11:36+00:00,2026-05-31T01:41:36+00:00,1800,synthetic_maintenance,deterministic synthetic Layer D assumption
9,BLK_GATEWAY_0005_02,GATEWAY_0005,"Alfouvar de Cima, Portugal",2026-05-31T03:14:45+00:00,2026-05-31T03:44:45+00:00,1800,synthetic_maintenance,deterministic synthetic Layer D assumption


## Enrich Visibility Windows

Each visibility window receives station capacity, antenna type, reconfiguration time, availability bounds, and any overlapping blackout IDs. Windows that overlap a blackout are retained but flagged as not operationally available, which keeps the modeling assumption visible for review.

In [16]:
def find_overlapping_blackouts(
    station_id: str,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    blackouts_by_station: dict[str, pd.DataFrame],
) -> str:
    """Return semicolon-separated blackout IDs that overlap one visibility window.

    Parameters
    ----------
    station_id:
        Gateway station identifier for the visibility window.
    window_start:
        Visibility-window start time.
    window_end:
        Visibility-window end time.
    blackouts_by_station:
        Dictionary of blackout DataFrames keyed by station ID.

    Returns
    -------
    str
        Semicolon-separated blackout IDs, or an empty string when none overlap.
    """
    station_blackouts = blackouts_by_station.get(station_id)
    if station_blackouts is None or station_blackouts.empty:
        return ""

    overlaps = station_blackouts[
        (window_start < station_blackouts["blackout_end_ts"])
        & (window_end > station_blackouts["blackout_start_ts"])
    ]
    return ";".join(overlaps["blackout_id"].tolist())


def enrich_visibility_windows(
    windows_df: pd.DataFrame,
    stations_df: pd.DataFrame,
    blackouts_df: pd.DataFrame,
    availability_start: pd.Timestamp,
    availability_end: pd.Timestamp,
) -> pd.DataFrame:
    """Add deterministic operational scheduling constraints to visibility windows.

    Parameters
    ----------
    windows_df:
        Raw Layer C visibility windows.
    stations_df:
        Station-level operational assumptions.
    blackouts_df:
        Synthetic station blackout windows.
    availability_start:
        Start of the station availability horizon.
    availability_end:
        End of the station availability horizon.

    Returns
    -------
    pandas.DataFrame
        Enriched visibility-window table with operational constraint fields.
    """
    blackouts = blackouts_df.copy()
    blackouts["blackout_start_ts"] = pd.to_datetime(blackouts["blackout_start"], utc=True)
    blackouts["blackout_end_ts"] = pd.to_datetime(blackouts["blackout_end"], utc=True)
    blackouts_by_station = {
        station_id: group.reset_index(drop=True)
        for station_id, group in blackouts.groupby("station_id")
    }

    enriched = windows_df.merge(stations_df, on=["station_id", "station_name"], how="left")
    missing_station_assumptions = enriched["antenna_type"].isna().sum()
    if missing_station_assumptions:
        raise ValueError(f"Missing station assumptions for {missing_station_assumptions} visibility windows.")

    enriched["availability_start"] = availability_start.isoformat()
    enriched["availability_end"] = availability_end.isoformat()
    enriched["blackout_window_ids"] = [
        find_overlapping_blackouts(row.station_id, row.start_time, row.end_time, blackouts_by_station)
        for row in enriched.itertuples(index=False)
    ]
    enriched["overlaps_blackout"] = enriched["blackout_window_ids"].ne("")
    enriched["is_operationally_available"] = ~enriched["overlaps_blackout"]
    enriched["operational_constraint_source"] = "deterministic synthetic Layer D assumptions"

    output_columns = [
        "window_id",
        "satellite_name",
        "norad_id",
        "station_id",
        "station_name",
        "start_time",
        "end_time",
        "duration_seconds",
        "max_elevation_deg",
        "source_epoch",
        "antenna_type",
        "station_capacity",
        "reconfiguration_time_seconds",
        "availability_start",
        "availability_end",
        "blackout_window_ids",
        "overlaps_blackout",
        "is_operationally_available",
        "operational_constraint_source",
    ]
    return enriched[output_columns]


enriched_windows = enrich_visibility_windows(
    visibility_windows,
    stations,
    blackout_windows,
    horizon_start,
    horizon_end,
)

print(f"Enriched visibility windows: {len(enriched_windows)}")
print(f"Operationally available windows: {int(enriched_windows['is_operationally_available'].sum())}")
print(f"Windows overlapping blackouts: {int(enriched_windows['overlaps_blackout'].sum())}")
enriched_windows.head(10)

Enriched visibility windows: 4091
Operationally available windows: 2922
Windows overlapping blackouts: 1169


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch,antenna_type,station_capacity,reconfiguration_time_seconds,availability_start,availability_end,blackout_window_ids,overlaps_blackout,is_operationally_available,operational_constraint_source
0,VIS_000001,STARLINK-1296,45412,GATEWAY_0001,"Adelanto, CA",2026-05-31 00:00:00+00:00,2026-05-31 00:06:00+00:00,360,56.652,2026-06-17T20:32:56Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
1,VIS_000002,STARLINK-1068,44772,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,17.071,2026-06-18T04:25:21Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
2,VIS_000003,STARLINK-1277,45369,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,10.446,2026-06-17T23:46:21Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
3,VIS_000004,STARLINK-1300,45374,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:03:00+00:00,180,38.754,2026-06-18T01:45:28Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
4,VIS_000005,STARLINK-1167,45071,GATEWAY_0003,"Akita, Japan",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,11.375,2026-06-18T02:34:02Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
5,VIS_000006,STARLINK-1154,45198,GATEWAY_0003,"Akita, Japan",2026-05-31 00:00:00+00:00,2026-05-31 00:04:00+00:00,240,21.188,2026-06-18T04:49:37Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
6,VIS_000007,STARLINK-1269,45197,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:05:00+00:00,300,33.113,2026-06-17T14:51:53Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
7,VIS_000008,STARLINK-1302,45375,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:02:00+00:00,120,20.103,2026-06-17T23:12:53Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
8,VIS_000009,STARLINK-1297,45395,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,10.072,2026-06-18T05:00:33Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
9,VIS_000010,STARLINK-1350,45535,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:03:00+00:00,180,13.082,2026-06-17T12:25:52Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions


## Validation

These checks confirm that Layer D added the expected operational fields, preserved the raw visibility-window row count, produced valid blackout intervals, and assigned positive resource constraints.

In [17]:
def validate_blackout_table(blackouts_df: pd.DataFrame) -> dict:
    """Validate the synthetic blackout-window table.

    Parameters
    ----------
    blackouts_df:
        Blackout table generated by Layer D.

    Returns
    -------
    dict
        Counts for missing values, duplicate IDs, and invalid timing.
    """
    required_columns = ["blackout_id", "station_id", "station_name", "blackout_start", "blackout_end", "duration_seconds", "reason", "source"]
    missing_columns = [column for column in required_columns if column not in blackouts_df.columns]
    if missing_columns:
        raise ValueError(f"Missing required blackout columns: {missing_columns}")

    starts = pd.to_datetime(blackouts_df["blackout_start"], utc=True)
    ends = pd.to_datetime(blackouts_df["blackout_end"], utc=True)
    sorted_blackouts = blackouts_df.assign(_start=starts, _end=ends).sort_values(["station_id", "_start"])
    overlapping_blackouts = 0
    for _, group in sorted_blackouts.groupby("station_id"):
        overlapping_blackouts += int((group["_start"].iloc[1:].to_numpy() < group["_end"].iloc[:-1].to_numpy()).sum())

    report = {
        "rows": len(blackouts_df),
        "missing_blackout_ids": int(blackouts_df["blackout_id"].fillna("").str.strip().eq("").sum()),
        "duplicate_blackout_ids": int(blackouts_df["blackout_id"].duplicated().sum()),
        "invalid_time_order": int((ends <= starts).sum()),
        "non_positive_durations": int((blackouts_df["duration_seconds"] <= 0).sum()),
        "overlapping_blackouts_same_station": overlapping_blackouts,
    }
    failed_checks = {name: value for name, value in report.items() if name != "rows" and value != 0}
    if failed_checks:
        raise ValueError(f"Blackout validation failed: {failed_checks}")
    return report


def validate_enriched_windows(raw_windows_df: pd.DataFrame, enriched_windows_df: pd.DataFrame) -> dict:
    """Validate the operationally enriched visibility-window table.

    Parameters
    ----------
    raw_windows_df:
        Original Layer C visibility windows.
    enriched_windows_df:
        Layer D enriched windows.

    Returns
    -------
    dict
        Counts for missing fields, invalid constraints, and row preservation.
    """
    required_columns = [
        "antenna_type",
        "station_capacity",
        "reconfiguration_time_seconds",
        "availability_start",
        "availability_end",
        "blackout_window_ids",
        "overlaps_blackout",
        "is_operationally_available",
        "operational_constraint_source",
    ]
    missing_columns = [column for column in required_columns if column not in enriched_windows_df.columns]
    if missing_columns:
        raise ValueError(f"Missing required enriched columns: {missing_columns}")

    report = {
        "rows": len(enriched_windows_df),
        "row_count_difference_from_layer_c": len(enriched_windows_df) - len(raw_windows_df),
        "missing_antenna_type": int(enriched_windows_df["antenna_type"].fillna("").str.strip().eq("").sum()),
        "invalid_station_capacity": int((enriched_windows_df["station_capacity"] < 1).sum()),
        "invalid_reconfiguration_time": int((enriched_windows_df["reconfiguration_time_seconds"] < 0).sum()),
        "missing_constraint_source": int(enriched_windows_df["operational_constraint_source"].fillna("").str.strip().eq("").sum()),
    }
    failed_checks = {name: value for name, value in report.items() if name not in ["rows"] and value != 0}
    if failed_checks:
        raise ValueError(f"Enriched-window validation failed: {failed_checks}")
    return report


blackout_validation = validate_blackout_table(blackout_windows)
enriched_validation = validate_enriched_windows(visibility_windows, enriched_windows)

print("Blackout validation:")
print(blackout_validation)
print("\nEnriched-window validation:")
print(enriched_validation)
print("\nAvailability counts:")
print(enriched_windows["is_operationally_available"].value_counts().to_string())
print("\nPreview:")
enriched_windows.head(10)

Blackout validation:
{'rows': 100, 'missing_blackout_ids': 0, 'duplicate_blackout_ids': 0, 'invalid_time_order': 0, 'non_positive_durations': 0, 'overlapping_blackouts_same_station': 0}

Enriched-window validation:
{'rows': 4091, 'row_count_difference_from_layer_c': 0, 'missing_antenna_type': 0, 'invalid_station_capacity': 0, 'invalid_reconfiguration_time': 0, 'missing_constraint_source': 0}

Availability counts:
True     2922
False    1169

Preview:


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch,antenna_type,station_capacity,reconfiguration_time_seconds,availability_start,availability_end,blackout_window_ids,overlaps_blackout,is_operationally_available,operational_constraint_source
0,VIS_000001,STARLINK-1296,45412,GATEWAY_0001,"Adelanto, CA",2026-05-31 00:00:00+00:00,2026-05-31 00:06:00+00:00,360,56.652,2026-06-17T20:32:56Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
1,VIS_000002,STARLINK-1068,44772,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,17.071,2026-06-18T04:25:21Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
2,VIS_000003,STARLINK-1277,45369,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,10.446,2026-06-17T23:46:21Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
3,VIS_000004,STARLINK-1300,45374,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:03:00+00:00,180,38.754,2026-06-18T01:45:28Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
4,VIS_000005,STARLINK-1167,45071,GATEWAY_0003,"Akita, Japan",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,11.375,2026-06-18T02:34:02Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
5,VIS_000006,STARLINK-1154,45198,GATEWAY_0003,"Akita, Japan",2026-05-31 00:00:00+00:00,2026-05-31 00:04:00+00:00,240,21.188,2026-06-18T04:49:37Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
6,VIS_000007,STARLINK-1269,45197,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:05:00+00:00,300,33.113,2026-06-17T14:51:53Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
7,VIS_000008,STARLINK-1302,45375,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:02:00+00:00,120,20.103,2026-06-17T23:12:53Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
8,VIS_000009,STARLINK-1297,45395,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,10.072,2026-06-18T05:00:33Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
9,VIS_000010,STARLINK-1350,45535,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:03:00+00:00,180,13.082,2026-06-17T12:25:52Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions


## Save Phase 4 Outputs

The blackout table records the synthetic unavailability assumptions. The enriched visibility-window table keeps all Layer C windows while marking which ones are operationally available under those assumptions.

In [18]:
blackout_windows.to_csv(BLACKOUT_OUTPUT_CSV, index=False)
enriched_windows.to_csv(ENRICHED_OUTPUT_CSV, index=False)

print(f"Saved synthetic blackout windows to: {BLACKOUT_OUTPUT_CSV.resolve()}")
print(f"Blackout rows saved: {len(blackout_windows)}")
print(f"Saved enriched visibility windows to: {ENRICHED_OUTPUT_CSV.resolve()}")
print(f"Enriched rows saved: {len(enriched_windows)}")
enriched_windows.head(10)

Saved synthetic blackout windows to: C:\Users\felip\Documents\Polimi\Semester 3\SE2 Quantum Satisfiability Project\satelite-optimization-quantum-annealing\PoliMi-SWE2-research-project\data\dataset\starlink\synthetic_gateway_blackouts.csv
Blackout rows saved: 100
Saved enriched visibility windows to: C:\Users\felip\Documents\Polimi\Semester 3\SE2 Quantum Satisfiability Project\satelite-optimization-quantum-annealing\PoliMi-SWE2-research-project\data\dataset\starlink\starlink_visibility_windows_operational.csv
Enriched rows saved: 4091


,window_id,satellite_name,norad_id,station_id,station_name,start_time,end_time,duration_seconds,max_elevation_deg,source_epoch,antenna_type,station_capacity,reconfiguration_time_seconds,availability_start,availability_end,blackout_window_ids,overlaps_blackout,is_operationally_available,operational_constraint_source
0,VIS_000001,STARLINK-1296,45412,GATEWAY_0001,"Adelanto, CA",2026-05-31 00:00:00+00:00,2026-05-31 00:06:00+00:00,360,56.652,2026-06-17T20:32:56Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
1,VIS_000002,STARLINK-1068,44772,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,17.071,2026-06-18T04:25:21Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
2,VIS_000003,STARLINK-1277,45369,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,10.446,2026-06-17T23:46:21Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
3,VIS_000004,STARLINK-1300,45374,GATEWAY_0002,"Aerzen, Germany",2026-05-31 00:00:00+00:00,2026-05-31 00:03:00+00:00,180,38.754,2026-06-18T01:45:28Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
4,VIS_000005,STARLINK-1167,45071,GATEWAY_0003,"Akita, Japan",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,11.375,2026-06-18T02:34:02Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
5,VIS_000006,STARLINK-1154,45198,GATEWAY_0003,"Akita, Japan",2026-05-31 00:00:00+00:00,2026-05-31 00:04:00+00:00,240,21.188,2026-06-18T04:49:37Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
6,VIS_000007,STARLINK-1269,45197,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:05:00+00:00,300,33.113,2026-06-17T14:51:53Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
7,VIS_000008,STARLINK-1302,45375,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:02:00+00:00,120,20.103,2026-06-17T23:12:53Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
8,VIS_000009,STARLINK-1297,45395,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:01:00+00:00,60,10.072,2026-06-18T05:00:33Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions
9,VIS_000010,STARLINK-1350,45535,GATEWAY_0004,"Akron, OH",2026-05-31 00:00:00+00:00,2026-05-31 00:03:00+00:00,180,13.082,2026-06-17T12:25:52Z,dish,1,180,2026-05-31T00:00:00+00:00,2026-05-31T04:00:00+00:00,,False,True,deterministic synthetic Layer D assumptions


## Phase 4 Result

Phase 4 produces visibility windows enriched with deterministic operational constraints. These records are ready for a later Layer E notebook to assign mission-like priorities, urgencies, and scheduling weights after review.